In [28]:
# packages and working directory  
import scipy 
import sklearn
import econml 
import arch
import os 
import torch
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
from scipy import stats
from scipy.stats import ttest_ind
from scipy.optimize import approx_fprime
from time import time
from torch_choice.data import ChoiceDataset, utils
from torch_choice import run
from torch_choice.data import ChoiceDataset
from torch_choice.model import ConditionalLogitModel
from torch.utils.data import DataLoader
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme import models
from biogeme import models, database
from biogeme.expressions import Beta, Variable
# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)



In [29]:
df = pd.read_csv('4choicestalong.csv')
df.describe()

,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,il_bsarg_63,il_bsarg_64,il_bsarg_70,scenario,original_scenario,lhw_0,lhw_1,lhw_2,lhw_3,choice_made
count,1.718000e+04,1.718000e+04,1.718000e+04,1.718000e+04,1.718000e+04,1.718000e+04,1.718000e+04,17180.000000,17180.000000,17180.000000,...,17180.000000,17180.000000,17180.000000,17180.000000,17180.000000,17180.0,17180.000000,17180.000000,17180.000000,17180.000000
mean,4.218959e+06,4.218959e+08,7.474154e+07,4.769461e+07,5.971600e+07,4.218959e+06,4.218959e+08,48.288475,0.442142,0.227939,...,1878.919483,1911.016011,1897.955814,1.500000,1.897322,0.0,14.140861,38.638882,61.454249,0.250000
std,1.559470e+06,1.559470e+08,1.721504e+08,1.419162e+08,1.560383e+08,1.559470e+06,1.559470e+08,11.711231,0.496656,1.097796,...,2033.442702,2032.692573,2030.590588,1.118067,0.622776,0.0,4.571384,3.512094,4.879186,0.433025
min,8.701000e+05,8.701000e+07,0.000000e+00,0.000000e+00,0.000000e+00,8.701000e+05,8.701000e+07,19.000000,0.000000,0.000000,...,-730.070000,0.000000,-300.000000,0.000000,0.000000,0.0,1.000000,27.000000,49.000000,0.000000
25%,2.773500e+06,2.773500e+08,0.000000e+00,0.000000e+00,0.000000e+00,2.773500e+06,2.773500e+08,40.000000,0.000000,0.000000,...,469.930000,490.765000,471.857500,0.750000,2.000000,0.0,11.000000,37.000000,59.000000,0.000000
50%,4.813100e+06,4.813100e+08,0.000000e+00,0.000000e+00,0.000000e+00,4.813100e+06,4.813100e+08,48.000000,0.000000,0.000000,...,1312.715000,1350.615000,1336.720000,1.500000,2.000000,0.0,14.000000,40.000000,61.000000,0.000000
75%,5.550400e+06,5.550400e+08,0.000000e+00,0.000000e+00,0.000000e+00,5.550400e+06,5.550400e+08,56.000000,1.000000,0.000000,...,2626.717500,2662.195000,2650.850000,2.250000,2.000000,0.0,17.000000,40.000000,65.000000,0.250000
max,6.201200e+06,6.201200e+08,6.196200e+08,6.193200e+08,6.186300e+08,6.201200e+06,6.201200e+08,85.000000,1.000000,6.000000,...,35131.750000,35131.750000,35131.750000,3.000000,3.000000,0.0,26.000000,48.000000,75.000000,1.000000


In [30]:
df1 = pd.read_csv('filtered.csv')
def map_lhw(row):
    # Ensure the scenario prefix 'h' is included when constructing the column name
    scenario_prefix = 'h' if not row["scenario"].startswith('h') else ''
    scenario_column_name = f'lhw_{scenario_prefix}{row["scenario"]}'
    return row[scenario_column_name]

# Apply the corrected function
df1['lhw_scenario'] = df1.apply(map_lhw, axis=1)

# Print a sample to verify the column has been created correctly
print(df1[['idperson', 'scenario', 'lhw_scenario']].head())
df1 = df1.sort_values(by='idperson')
df1.head()

   idperson scenario  lhw_scenario
0  87010001       h0             0
1  87060001       h0             0
2  87070001       h0             0
3  87090001       h0             0
4  87170002       h0             0


,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,il_bsarg_64,il_bsarg_70,scenario,original_scenario,choice_made,lhw_h0,lhw_h1,lhw_h2,lhw_h3,lhw_scenario
0,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,0.00,0.00,h0,h2,0,0,7,42,54,0
9160,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,2360.64,2360.64,h2,h2,1,0,7,42,54,42
4580,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,353.44,353.44,h1,h2,0,0,7,42,54,7
13740,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,3035.10,3035.10,h3,h2,0,0,7,42,54,54
9161,870600,87060001,0,0,0,870600,87060001,32,1,0,...,3043.63,3043.63,h2,h2,1,0,11,44,50,44


In [31]:
# Identify individuals with any negative 'ils_udb_yds' values
negative_c_ids = df1[df1['ils_udb_yds'] <= 250]['idperson'].unique()
negative_l_ids = df1[df1['lhw'] >77 ]['idperson'].unique()

# Check how many individuals are affected
print(f"Number of individuals with negative leisure: {len(negative_l_ids)}")

# Check how many individuals are affected
print(f"Number of individuals with negative consumption: {len(negative_c_ids)}")
# Filter long_df to exclude all rows belonging to individuals identified in step 1
df1_filt = df1[~df1['idperson'].isin(negative_c_ids)]
df1_filt = df1_filt [~ df1_filt ['idperson'].isin(negative_l_ids)]
# Verify the removal
print(f"Original dataframe size: {df1.shape}")
print(f"Filtered dataframe size: {df1_filt.shape}")


Number of individuals with negative leisure: 3
Number of individuals with negative consumption: 272
Original dataframe size: (18320, 350)
Filtered dataframe size: (17220, 350)


In [32]:
df2 = df1_filt.copy()



# Create the 'labor' column based on the 'scenario' column
df2['labor'] = df2['lhw_scenario']

df2['log_y'] = np.log(df2['ils_udb_yds'])
df2['log_l'] = np.log(80 - df2['labor'])
df2['log2_y'] = df2['log_y']**2 
df2['log2_l'] =  df2['log_l']**2
df2['log_y_l'] = df2['log_y'] * df2['log_l']

In [33]:
df2['log_l'].mean()

3.90489046240276

In [17]:
df2.columns.tolist()

['idhh',
 'idperson',
 'idmother',
 'idfather',
 'idpartner',
 'idorighh',
 'idorigperson',
 'dag',
 'dgn',
 'dec',
 'dwt',
 'dms',
 'deh',
 'drgn2',
 'ddi',
 'dlg_s',
 'ddilv',
 'dcz',
 'drgur',
 'drgmd',
 'drgru',
 'ddt',
 'dsu00',
 'dsu01',
 'dsu02',
 'dncsy',
 'dmb',
 'dct',
 'dehde',
 'dey',
 'drgn1',
 'dsr',
 'les',
 'loc',
 'loopcount_pens',
 'liwft_s',
 'lhw',
 'lindi',
 'lhwsr_s',
 'lhwsesr_s',
 'lunmy_s',
 'lunmy',
 'liwmy_s',
 'liwwh',
 'liwmy_a',
 'lnu',
 'lhwpv_a',
 'liwmy02_a',
 'lfs',
 'lcs',
 'liwmy',
 'liwftmy',
 'liwptmy',
 'lpemy',
 'lse',
 'liwmy_f',
 'lhw_f',
 'liwwh_f',
 'lunmy_f',
 'yem',
 'yse',
 'yemmc_s',
 'yiy',
 'yot',
 'ypr',
 'ypt',
 'ypp',
 'yemxm_s',
 'yemmy',
 'ysemy',
 'yemmw_s',
 'ysemw_s',
 'ysemc_s',
 'yempv_s',
 'yivwg',
 'yempv_a',
 'ysv',
 'yptmp',
 'yds',
 'ydses_o',
 'poa00',
 'pdi00',
 'pdicm',
 'pdinc',
 'psuwd00',
 'poacm',
 'poanc',
 'poaot',
 'poa',
 'pdiot',
 'pdi',
 'psuwdcm',
 'psuot',
 'psu',
 'poanc00_s',
 'poancna_s',
 'poancrg_s',
 

In [13]:
if torch.cuda.is_available():
    print(f'CUDA device used: {torch.cuda.get_device_name()}')
    device = 'cuda'
else:
    print('Running tutorial on CPU.')
    device = 'cpu'
item_index = df2[df2['choice_made'] == 1].sort_values(by='idperson')['scenario'].reset_index(drop=True)
print(item_index)
item_names = ['h0', 'h1', 'h2', 'h3']
num_items = 4
encoder = dict(zip(item_names, range(num_items)))
print(f"{encoder=:}")
item_index = item_index.map(lambda x: encoder[x])
item_index = torch.LongTensor(item_index)
print(f"{item_index=:}")

Running tutorial on CPU.
0       h2
1       h2
2       h0
3       h2
4       h2
        ..
4300    h2
4301    h0
4302    h2
4303    h2
4304    h3
Name: scenario, Length: 4305, dtype: object
encoder={'h0': 0, 'h1': 1, 'h2': 2, 'h3': 3}
item_index=tensor([2, 2, 0,  ..., 2, 2, 3])


In [23]:
import torch
from torch_choice.data import ChoiceDataset

# Define features to be used in the model
features = ['log_y', 'log_l', 'log2_y', 'log2_l', 'log_y_l']
# Add demographic variables if you want to include them in the model
features += ['dag', 'dgn', 'deh']

# Convert features to tensors
X = torch.tensor(df2[features].values, dtype=torch.float32)
# Choice made (convert scenarios to indices and then to tensor)
choice = torch.LongTensor(item_index)

# Create dataset
dataset = ChoiceDataset(X=X, item_index=choice, num_items=num_items)


No `session_index` is provided, assume each choice instance is in its own session.


In [26]:
from torch_choice.model import ConditionalLogitModel

# Instantiate the model
model = ConditionalLogitModel(num_items=num_items).to(device)


ValueError: Either coef_variation_dict or formula should be provided to specify the model.